## Initialisation

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Fonction pour transformer les codes en libellés complets en utilisant les métadonnées
def map_metadata(df, df_meta, list_vars):
    for var in list_vars:
        var_dict = df_meta[df_meta['COD_VAR'] == var].set_index('COD_MOD')['LIB_MOD'].to_dict()
    
        df[var] = df[var].map(var_dict)
    return df


# Fonction pour afficher les doublons et valeurs manquantes
def count_null_and_duplicates(df):
    duplicates_count = df.duplicated().sum()

    if duplicates_count > 0:
        display(df[df.duplicated()])
    else:
        print("--- Aucun doublon")

    null_values_count = df.isnull().sum()
    missing_data = null_values_count[null_values_count > 0]

    if missing_data.empty:
        print('\n--- Aucune valeur manquante')
    else:
        results = pd.DataFrame({
            'Valeurs manquantes (nb)': missing_data,
            'Valeurs manquantes (%)': round((missing_data / len(df)) * 100, 2)
        })
        display(results)




## Chargement dataset - Salaires privés moyens

In [ ]:
# Chargement données + métadonnées
salaries_csv = pd.read_csv('src_insee/DS_BTS_SAL_EQTP_SEX_AGE_2023_CSV_FR/DS_BTS_SAL_EQTP_SEX_AGE_2023_data.csv', sep=';')
df_salaries = pd.DataFrame(salaries_csv)

salaries_meta_csv = pd.read_csv('src_insee/DS_BTS_SAL_EQTP_SEX_AGE_2023_CSV_FR/DS_BTS_SAL_EQTP_SEX_AGE_2023_metadata.csv', sep=';')
df_salaries_meta = pd.DataFrame(salaries_meta_csv)


# Suppression des lignes et colonnes inutiles
df_salaries = df_salaries[(df_salaries['TIME_PERIOD'] == 2022) & (df_salaries['GEO_OBJECT'] == 'COM')]
df_salaries = df_salaries.drop(['TIME_PERIOD', 'DERA_MEASURE', 'GEO_OBJECT', 'FREQ', 'CONF_STATUS'], axis=1)


# Aperçu du dataframe
display(df_salaries)


# Doublons et valeurs manquantes
count_null_and_duplicates(df_salaries)


# Transformation des colonnes
# df_salaries = map_metadata(df_salaries, df_salaries_meta, ['GEO', 'SEX', 'AGE'])

In [ ]:
# Analysons la distribution du salaire avant imputation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : Histogramme du salaire
axes[0].hist(df_salaries['OBS_VALUE'].dropna(), bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Salaire')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution du salaire (avant imputation)')
axes[0].grid(alpha=0.3)

# Graphique 2 : Boxplot de l'âge par genre (avec seaborn)
sns.boxplot(data=df_salaries, x='SEX', y='OBS_VALUE', ax=axes[1])
axes[1].set_xlabel('Genre')
axes[1].set_ylabel('Salaire')
axes[1].set_title('Salaire moyen par genre')

plt.tight_layout()
plt.show()

# Calculons le salaire médian par genre
print("Salaire médian par genre :")
print(df_salaries.groupby('SEX')['OBS_VALUE'].median())



# Analysons la distribution du salaire avant imputation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : Histogramme du salaire
axes[0].hist(df_salaries['OBS_VALUE'].dropna(), bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Salaire')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution du salaire (avant imputation)')
axes[0].grid(alpha=0.3)

# Graphique 2 : Boxplot de l'âge par genre (avec seaborn)
sns.boxplot(data=df_salaries, x='AGE', y='OBS_VALUE', ax=axes[1])
axes[1].set_xlabel('Tranche d\'âge')
axes[1].set_ylabel('Salaire')
axes[1].set_title('Salaire moyen par tranche d\'âge')

plt.tight_layout()
plt.show()

# Calculons le salaire médian par genre
print("Salaire médian par tranche d\'âge :")
print(df_salaries.groupby('AGE')['OBS_VALUE'].median())

In [ ]:
df_salaries['OBS_VALUE'] = df_salaries['OBS_VALUE'].fillna(
    df_salaries.groupby(['SEX', 'AGE'])['OBS_VALUE'].transform('median')
)

display(df_salaries)
count_null_and_duplicates(df_salaries)

Données salariales :
* Salaire moyen total
* Salaire moyen des femmes
* Salaire moyen des hommes
* Salaire moyen des -25 ans
* Salaire moyen des 25-39 ans
* Salaire moyen des 40-49 ans
* Salaire moyen des 50-54 ans
* Salaire moyen des +55 ans

## Chargement dataset - Population

In [ ]:
# Chargement données + métadonnées
population_csv = pd.read_csv('src_insee/DS_RP_POPULATION_COMP_2022_CSV_FR/DS_RP_POPULATION_COMP_2022_data.csv', sep=';')
df_population = pd.DataFrame(population_csv)

population_meta_csv = pd.read_csv('src_insee/DS_RP_POPULATION_COMP_2022_CSV_FR/DS_RP_POPULATION_COMP_2022_metadata.csv', sep=';')
df_population_meta = pd.DataFrame(population_meta_csv)


# Suppression des lignes et colonnes inutiles
df_population = df_population[(df_population['TIME_PERIOD'] == 2022) & (df_population['GEO_OBJECT'] == 'COM')]
df_population = df_population.drop(['RP_MEASURE', 'TIME_PERIOD', 'GEO_OBJECT'], axis=1)


# Aperçu du dataframe
display(df_population)


# Doublons et valeurs manquantes
count_null_and_duplicates(df_population)


# Transformation des colonnes
# df_population = map_metadata(df_population, df_population_meta, ['GEO', 'AGE', 'SEX', 'PCS'])

Données de population :
* Population totale
* Nombre d'hommes
* Nombre de femmes
* Population de 15-24 ans
* Population 25-54 ans
* Population +55 ans
* Population PCS Agriculteurs
* Population PCS Artisans, commerçants et chefs d’entreprise
* Population PCS Cadres et professions intellectuelles supérieures
* Population PCS Professions intermédiaires
* Population PCS Employés
* Population PCS Ouvriers
* Population PCS Autres inactifs
* Population PCS Retraités
* Population PCS Etudiants ou élèves
* Population PCS Total (équivalent à population -18 ans ?)

## Chargement dataset - Conjugalité

In [ ]:
# Chargement données + métadonnées
conjugality_csv = pd.read_csv('src_insee/DS_RP_MENAGES_PRINC_2022_CSV_FR/DS_RP_MENAGES_PRINC_2022_data.csv', sep=';')
df_conjugality = pd.DataFrame(conjugality_csv)

conjugality_meta_csv = pd.read_csv('src_insee/DS_RP_MENAGES_PRINC_2022_CSV_FR/DS_RP_MENAGES_PRINC_2022_metadata.csv', sep=';')
df_conjugality_meta = pd.DataFrame(conjugality_meta_csv)


# Suppression des lignes et colonnes inutiles
df_conjugality = df_conjugality[(df_conjugality['TIME_PERIOD'] == 2022) & (df_conjugality['GEO_OBJECT'] == 'COM') & (df_conjugality['RP_MEASURE'] == 'POP') & (df_conjugality['NOC'] == '_T') & (df_conjugality['OCS'] == '_T')]
df_conjugality = df_conjugality.drop(['FREQ', 'TIME_PERIOD', 'GEO_OBJECT', 'RP_MEASURE', 'OCS', 'NOC'], axis=1)


# Aperçu du dataframe
display(df_conjugality)


# Doublons et valeurs manquantes
count_null_and_duplicates(df_conjugality)


# Transformation des colonnes
# df_conjugality = map_metadata(df_conjugality, df_conjugality_meta, ['GEO', 'AGE', 'CIVIL_STATUS', 'COUPLE'])

## Chargement dataset - Ménages

In [ ]:
# Chargement données + métadonnées
household_csv = pd.read_csv('src_insee/DS_RP_MENAGES_COMP_2022_CSV_FR/DS_RP_MENAGES_COMP_2022_data.csv', sep=';')
df_household = pd.DataFrame(household_csv)

household_meta_csv = pd.read_csv('src_insee/DS_RP_MENAGES_COMP_2022_CSV_FR/DS_RP_MENAGES_COMP_2022_metadata.csv', sep=';')
df_household_meta = pd.DataFrame(household_meta_csv)


# Suppression des lignes et colonnes inutiles
df_household = df_household[(df_household['TIME_PERIOD'] == 2022) & (df_household['GEO_OBJECT'] == 'COM') & (df_household['PREFPH'] == '_T') & (df_household['RP_MEASURE'] == 'DWELLINGS_POPSIZE')]
df_household = df_household.drop(['TIME_PERIOD', 'GEO_OBJECT', 'FREQ', 'OCS', 'PREFPH', 'RP_MEASURE'], axis=1)


# Aperçu du dataframe
display(df_household)


# Doublons et valeurs manquantes
count_null_and_duplicates(df_household)


# Transformation des colonnes
# df_household = map_metadata(df_household, df_household_meta, ['GEO', 'PCS', 'TPH'])